[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Impact-Map/opm_ehri_data/blob/va-section505-vs-opm-analysis/va_section505_vs_opm.ipynb)

*Running on Colab: uncomment the `!pip install` line in the Setup cell on first run.*

# VA MISSION Act §505 vs. OPM/EHRI — Reconciling Headcount and Flows

**Question that started this:** OPM's VA *headcount* runs slightly **above** the §505 onboard count (as expected, since §505 excludes certain staff). But OPM's *accessions/separations* looked **below** §505 — the opposite direction. Why would the two OPM datasets behave so differently against the same §505 baseline?

This notebook works it out against the live OPM/EHRI data (`impactproject/opm-ehri-data` on Hugging Face).

---

## Top line

1. **The flow "anomaly" was a comparison artifact, not a real divergence.** §505 reports accessions/separations as **quarterly cumulative** totals; OPM's dynamics files are **monthly**. Summed to the same fiscal quarter, OPM is *larger* than §505 on gross flows — same direction as headcount.

2. **There is a period/metric that aligns cleanly: the fiscal quarter, compared on _net_ change, over the §505-comparable population.** On that footing OPM and §505 agree to within ~3% (OPM net −1,375 vs §505 −1,421 for FY2026 Q2).

3. **Gross accessions/separations are *not* directly comparable.** OPM runs ~30% higher on both sides, and the inflation is *symmetric* (so net is preserved). Note: the OPM files log workforce **gains and losses by nature-of-action** — new hires, transfers, quits, retirements, RIF, terminations, deaths — they are **not** a complete log of every personnel action (there's no conversion/promotion category). The excess over §505 is mostly **permanent career staff**, not temp churn, and its exact driver (people-vs-actions counting, non-pay-status, period cutoffs) **cannot be fully decomposed from the published data** — see Part 4.

4. **The headcount gap is only ~half explained by §505's named exclusions.** Intermittent staff (~4,500) and OIG (~960) are demonstrably in OPM and not §505. But two named exclusions — Veterans Canteen Service and medical residents/interns/fellows — **aren't in OPM at all** (VCS is non-appropriated; residents are Without-Compensation appointments outside OPM's paid universe). The remaining ~6,600 of the gap is non-pay-status employees + snapshot-timing/definition differences.

**Recommendation for anyone comparing these:** use the fiscal quarter, compare *net* change, and match the population. Do not compare gross accessions/separations, and do not compare a single OPM month against a §505 quarter.

Source for §505 figures: https://department.va.gov/employees/va-mission-act-section-505-data/

## Caveats (read before quoting any number)

- **OPM dynamics data is revised.** Recent months are incomplete and get backfilled. February 2026 is visibly light (see Part 2). This notebook auto-selects the **latest available version** of each month, but a re-run weeks later may show higher Q2 flows.
- **Snapshot timing differs.** OPM employment is an end-of-period snapshot; the comparison below uses OPM **April** vs §505 **March** for the stock, which adds ~1 month of drift (~300 people at current trend).
- **§505 does not publish separations by reason**, so Part 4 shows OPM's composition and reasons about it; it is not a line-for-line category match.
- **"505-comparable" population** here = OPM VA minus the exclusions we can actually tag (OIG, intermittent, student-trainees). It does **not** remove non-pay-status employees (not a field we can isolate), so it is an approximation.

## Setup

In [1]:
# Colab / fresh env: uncomment to install
# !pip install -q huggingface_hub pandas pyarrow

import pandas as pd
from huggingface_hub import hf_hub_download, list_repo_files

REPO = "impactproject/opm-ehri-data"
pd.set_option("display.float_format", lambda x: f"{x:,.0f}")

# --- §505 published figures, FY2026 Q2 (Jan–Mar 2026), VA-wide ---
# https://department.va.gov/employees/va-mission-act-section-505-data/
S505 = {
    "headcount_march": 434_636,   # end-of-Q2 onboard
    "accessions_q2":     6_637,   # quarterly cumulative
    "separations_q2":    8_058,   # quarterly cumulative
}
S505["net_q2"] = S505["accessions_q2"] - S505["separations_q2"]  # -1,421
S505

/Users/abigailhaddad/Documents/repos/pull_usaspending/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'headcount_march': 434636,
 'accessions_q2': 6637,
 'separations_q2': 8058,
 'net_q2': -1421}

In [2]:
# Auto-detect the latest version of each (kind, YYYYMM) file on HF so this stays
# valid as OPM publishes revisions.
_ALL_FILES = list_repo_files(REPO, repo_type="dataset")

def latest_version(kind, ym):
    import re
    vs = []
    for f in _ALL_FILES:
        m = re.match(rf"{kind}/{kind}_{ym}_v(\d+)\.parquet", f)
        if m:
            vs.append(int(m.group(1)))
    if not vs:
        raise FileNotFoundError(f"no {kind} file for {ym}")
    return max(vs)

def load(kind, ym, columns=None):
    v = latest_version(kind, ym)
    path = hf_hub_download(REPO, f"{kind}/{kind}_{ym}_v{v}.parquet", repo_type="dataset")
    df = pd.read_parquet(path, columns=columns)
    df.attrs["version"] = v
    return df

def va_only(df):
    return df[df["agency_code"] == "VA"].copy()

MONTHS = ["202601", "202602", "202603", "202604"]  # Jan–Apr 2026
Q2     = ["202601", "202602", "202603"]            # FY2026 Q2 = Jan+Feb+Mar
print("latest versions:",
      {k: {ym: latest_version(k, ym) for ym in MONTHS}
       for k in ["employment", "accessions", "separations"]})

latest versions: {'employment': {'202601': 2, '202602': 2, '202603': 1, '202604': 1}, 'accessions': {'202601': 2, '202602': 2, '202603': 1, '202604': 1}, 'separations': {'202601': 2, '202602': 2, '202603': 1, '202604': 1}}


## Part 1 — Headcount reconciliation

Are the groups §505 says it excludes actually present in OPM, and do they account for OPM's higher headcount?

§505 excludes: OIG, Veterans Canteen Service, and several categories — intermittent staff, residents, interns, fellows, students, trainees, and non-pay-status employees.

In [3]:
emp = va_only(load("employment", "202604",
                   columns=["agency_code", "agency_subelement", "work_schedule",
                            "occupational_series", "pay_basis"]))
total = len(emp)

is_oig    = emp["agency_subelement"] == "INSPECTOR GENERAL"
is_interm = emp["work_schedule"] == "INTERMITTENT"
is_stud   = emp["occupational_series"].str.contains("STUDENT TRAINEE", na=False)
is_canteen = emp["agency_subelement"].str.contains("CANTEEN", case=False, na=False)

# de-duplicate so each person is removed once
oig_n    = int(is_oig.sum())
interm_n = int((is_interm & ~is_oig).sum())
stud_n   = int((is_stud & ~is_oig & ~is_interm).sum())
identifiable = oig_n + interm_n + stud_n

recon = pd.DataFrame([
    ("OPM VA total (April 2026)",            total,              ""),
    ("  − OIG (Inspector General)",          -oig_n,             "present in OPM"),
    ("  − Intermittent staff",              -interm_n,          "present in OPM"),
    ("  − Student trainees (net of above)", -stud_n,            "present in OPM"),
    ("  − Veterans Canteen Service",        -int(is_canteen.sum()), "NOT in OPM (non-appropriated)"),
    ("  − Residents/interns/fellows",        0,                 "NOT in OPM (Without-Compensation appts)"),
    ("OPM after identifiable exclusions",     total - identifiable, ""),
    ("§505 onboard (March 2026)",             S505["headcount_march"], ""),
    ("Residual gap (unexplained)",            total - identifiable - S505["headcount_march"], ""),
], columns=["line", "count", "note"])
recon

,line,count,note
0,OPM VA total (April 2026),446735,
1,− OIG (Inspector General),-956,present in OPM
2,− Intermittent staff,-4517,present in OPM
3,− Student trainees (net of above),-44,present in OPM
4,− Veterans Canteen Service,0,NOT in OPM (non-appropriated)
5,− Residents/interns/fellows,0,NOT in OPM (Without-Compensation appts)
6,OPM after identifiable exclusions,441218,
7,§505 onboard (March 2026),434636,
8,Residual gap (unexplained),6582,


**Read:** Intermittent (~4,500) and OIG (~960) are clearly in OPM and not in §505 — the exclusions are real and visible. But they cover only about **half** the ~12,100 raw gap. The two largest-sounding exclusions, Canteen Service and residents/interns/fellows, return **zero rows** — they were never in OPM, so they can't explain the gap. The remaining ~6,600 is non-pay-status employees plus snapshot-timing/definition differences (OPM April vs §505 March, differing as-of cutoffs, title-38 counting).

## Part 2 — Monthly headcount and flows (and the February gotcha)

Stocks (snapshots) and flows (accessions/separations) side by side, month by month.

In [4]:
def va_count(df):
    df = va_only(df)
    df["count"] = pd.to_numeric(df["count"])
    return df["count"].sum()

rows = []
prev_hc = None
for ym in MONTHS:
    hc  = len(va_only(load("employment", ym, columns=["agency_code"])))
    acc = va_count(load("accessions",  ym, columns=["agency_code", "count"]))
    sep = va_count(load("separations", ym, columns=["agency_code", "count"]))
    rows.append({
        "month": ym,
        "headcount": hc,
        "snapshot_change": "" if prev_hc is None else hc - prev_hc,
        "accessions": int(acc),
        "separations": int(sep),
        "net_flow": int(acc - sep),
    })
    prev_hc = hc

monthly = pd.DataFrame(rows)
monthly

,month,headcount,snapshot_change,accessions,separations,net_flow
0,202601,448422,,3934,4940,-1006
1,202602,446680,-1742,1090,809,281
2,202603,447021,341,4374,4905,-531
3,202604,446735,-286,3069,3223,-154


**Two things to notice:**

- **February is anomalously light** (~1,090 accessions / ~810 separations vs ~4,000–5,000 in Jan and Mar) yet headcount fell that period — a hallmark of incomplete, not-yet-revised dynamics data. Any period that includes February will undercount OPM flows.
- **Flows roughly track the snapshot change** but don't reconcile to the dollar (timing of snapshot date vs action effective date, plus non-pay-status). Over Jan→Apr the snapshot fell ~1,687 while cumulative net flow was ~−1,256 — same order of magnitude, flows running slightly light (consistent with the February gap).

## Part 3 — Flow reconciliation: the period/metric that aligns

Sum OPM to the **fiscal quarter** (FY2026 Q2 = Jan+Feb+Mar), and build a **§505-comparable** cut by removing the taggable exclusions from the flow files too.

In [5]:
FLOW_COLS = ["agency_code", "agency_subelement", "work_schedule",
             "occupational_series", "count"]

def comparable(va):
    """Drop the §505 exclusions we can tag: OIG, intermittent, student-trainees."""
    return va[(va["agency_subelement"] != "INSPECTOR GENERAL")
              & (va["work_schedule"] != "INTERMITTENT")
              & (~va["occupational_series"].str.contains("STUDENT TRAINEE", na=False))]

def q2_totals(kind):
    raw = comp = 0
    for ym in Q2:
        va = va_only(load(kind, ym, columns=FLOW_COLS))
        va["count"] = pd.to_numeric(va["count"])
        raw  += va["count"].sum()
        comp += comparable(va)["count"].sum()
    return int(raw), int(comp)

acc_raw, acc_comp = q2_totals("accessions")
sep_raw, sep_comp = q2_totals("separations")

flow = pd.DataFrame([
    ("Accessions",  acc_raw, acc_comp, S505["accessions_q2"]),
    ("Separations", sep_raw, sep_comp, S505["separations_q2"]),
    ("Net",         acc_raw - sep_raw, acc_comp - sep_comp, S505["net_q2"]),
], columns=["metric", "OPM_raw", "OPM_505comparable", "S505"])
flow["comp_pct_of_505"] = (flow["OPM_505comparable"] / flow["S505"] * 100).round(0)
flow

,metric,OPM_raw,OPM_505comparable,S505,comp_pct_of_505
0,Accessions,9398,9001,6637,136
1,Separations,10654,10376,8058,129
2,Net,-1256,-1375,-1421,97


**This is the punchline.**

- **Net change aligns:** OPM 505-comparable net (~−1,375) is within ~3% of §505's −1,421. Both say VA shrank by ~1,400 in the quarter. ✅
- **Gross flows do not align** even on the right period and population: OPM accessions ~136% and separations ~129% of §505. That gap is structural (see Part 4), not fixable by period or population choice.

**Why the original comparison looked backwards.** The starting puzzle was that OPM accessions/separations appeared to be only ~70–77% of §505 — smaller, not larger. That came from two mismatches stacked on top of each other:

1. **Period mismatch.** §505 publishes accessions/separations as **quarterly cumulative** totals (FY26 Q2 = Jan + Feb + Mar). OPM's dynamics files are **monthly**. Comparing one OPM month — or a partial quarter that includes the under-reported February — against a full §505 quarter makes OPM look far too small. Summed correctly to the quarter, OPM is *larger*, the same direction as headcount.

2. **Metric mismatch.** Even with the right period, *gross* accessions/separations mean different things in the two systems (Part 4). The number that is genuinely comparable is **net change**, and on net the two agree.

Put differently: the headcount comparison worked out of the box because a snapshot is a snapshot in both systems. The flow comparison only works once you (a) sum OPM to the fiscal quarter, (b) compare net rather than gross, and (c) match the population. Do all three and the apparent contradiction disappears.

## Part 4 — Why gross flows differ (and what we *can't* conclude)

The accessions/separations files contain **only** these categories — gains and losses to the workforce:

- **Accessions:** new hire (competitive / excepted / SES), transfer-in (individual / mass)
- **Separations:** quit, retirement (voluntary / early-out / other), RIF, termination (expired appt/other), transfer-out (individual / mass), other separation

There is **no** conversion, promotion, reassignment, or step-change category. So these files are **not** a complete log of every personnel action — a common misconception. They record who *entered* and *left* the workforce, by nature-of-action.

A tempting explanation for OPM's higher gross is "temp/seasonal churn that §505 excludes." The cells below test that — and it does not hold up.

In [6]:
frames = []
for ym in Q2:
    va = va_only(load("separations", ym,
                      columns=FLOW_COLS + ["separation_category"]))
    va["count"] = pd.to_numeric(va["count"])
    frames.append(comparable(va))
sep_q2 = pd.concat(frames)

by_cat = (sep_q2.groupby("separation_category")["count"].sum()
          .sort_values(ascending=False).rename("count").reset_index())
by_cat["pct"] = (by_cat["count"] / by_cat["count"].sum() * 100).round(1)
by_cat

,separation_category,count,pct
0,QUIT,5085,49
1,RETIREMENT - VOLUNTARY,3315,32
2,OTHER SEPARATION,1103,11
3,TERMINATION (EXPIRED APPT/OTHER),438,4
4,TRANSFER OUT - INDIVIDUAL TRANSFER,236,2
5,RETIREMENT - OTHER,172,2
6,RETIREMENT - EARLY OUT,26,0
7,TRANSFER OUT - MASS TRANSFER,1,0


In [7]:
# Test the "it's temp/seasonal churn" hypothesis: split the 505-comparable
# separations by permanent vs nonpermanent appointment.
frames = []
for ym in Q2:
    va = va_only(load("separations", ym,
                      columns=FLOW_COLS + ["appointment_type", "tenure"]))
    va["count"] = pd.to_numeric(va["count"])
    frames.append(comparable(va))
sep_appt = pd.concat(frames)

total   = int(sep_appt["count"].sum())
is_perm = (sep_appt["appointment_type"].str.contains("PERMANENT", na=False)
           & ~sep_appt["appointment_type"].str.contains("NONPERMANENT", na=False))
perm    = int(sep_appt[is_perm]["count"].sum())

print(f"OPM Q2 separations (505-comparable): {total:,}")
print(f"  permanent appointments:   {perm:,}  ({perm/total:.0%})")
print(f"  nonpermanent/temp/other:  {total-perm:,}  ({(total-perm)/total:.0%})")
print(f"\n§505 separations (Q2):       {S505['separations_q2']:,}")
print(f"OPM gross excess over §505:  {total - S505['separations_q2']:,}")
print("\n=> The excess is overwhelmingly PERMANENT career staff, not temp churn.")

OPM Q2 separations (505-comparable): 10,376
  permanent appointments:   10,058  (97%)
  nonpermanent/temp/other:  318  (3%)

§505 separations (Q2):       8,058
OPM gross excess over §505:  2,318

=> The excess is overwhelmingly PERMANENT career staff, not temp churn.


**The tempting explanations don't survive the data:**

- *It's not temp churn.* ~97% of OPM's Q2 separations are **permanent** appointments (Tenure Groups 1–2 — career and career-conditional staff actually leaving), not seasonal/temp turnover.
- *It's not OPM logging internal conversions.* The accession/separation files have **no** conversion or reassignment category, so OPM doesn't record those events as flows either.
- *It's not simply §505 excluding internal moves.* At the VA-wide level a purely internal VA move doesn't create a department-level accession/separation in OPM either, so that exclusion doesn't differentiate the two.

**What we *can* say with confidence:**

- OPM accessions and separations each run ~2,300 (≈30%) **above** §505 for the quarter, and the inflation is **symmetric** — both sides higher by nearly the same amount — which is exactly why **net change still reconciles** (OPM −1,375 vs §505 −1,421).
- That symmetry is the signature of a *matched in-and-out volume* that changes gross but not net. Plausible sources: §505 counting **distinct people** while OPM counts **personnel actions** (someone with two appointment actions in a quarter counts twice in OPM); §505's onboard base excluding **non-pay-status** employees whose actions OPM still records; or differing as-of / period cutoffs.

**What we *cannot* do from the published data:** decompose that ~2,300. §505 doesn't publish accessions/separations by appointment type, tenure, or its de-duplication rule, so the exact driver can't be pinned down here — it's a question for VA's §505 methodology documentation, not something these files can settle.

**Bottom line (unchanged and robust):** compare **net change over a fiscal quarter** and the two sources agree to within a few percent. Gross accessions/separations are not directly comparable, and we should not claim a specific mechanism for the gross gap that the data doesn't support.

## Conclusion

| Comparison | Verdict |
|---|---|
| Headcount (stock) | OPM > §505 by ~2.8%; exclusions explain ~half, rest is non-pay-status + timing |
| Gross accessions/separations | Not directly comparable — OPM ~30% higher on both sides (symmetric); exact driver undetermined from published data |
| **Net change over a fiscal quarter** | **Aligns to ~3% — use this** |

**How to compare OPM and §505 going forward:** match the **fiscal quarter** (sum OPM months), compare **net change**, and apply the same population exclusions. Don't compare gross flows, and don't put a single OPM month against a §505 quarter.

**Honest limit of this analysis:** OPM's gross flows run ~30% above §505 even after matching period and population, and the excess is permanent career staff rather than temp churn. We could *not* fully explain that gap from the published data — the leading candidates are counting conventions (personnel actions vs distinct people), non-pay-status coverage, and period cutoffs, but confirming which would require VA's §505 methodology detail. Net change is the metric that survives all of this.

*Data: `impactproject/opm-ehri-data` (Hugging Face). §505: https://department.va.gov/employees/va-mission-act-section-505-data/*